In [1]:
!pip install -q sentence-transformers chromadb

In [1]:
!pip install -U transformers torch

In [1]:
from pathlib import Path
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer

C:\Users\SAGNICK\anaconda3\envs\mypyref\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the document
file_path = Path("knowledge_text.txt")

text = file_path.read_text(encoding="utf-8")

print(text[:500])


Retrieval Augmented Generation (RAG)

Retrieval Augmented Generation is a technique that combines information retrieval with large language models. Instead of relying entirely on the knowledge stored in an LLM during training, RAG retrieves relevant information from an external knowledge source and provides it as context to the LLM. This helps the model answer questions using specific and potentially updated information.

Document Chunking

Document chunking is the process of splitting large do


In [3]:
# Split into chunks 
def create_chunks(text, chunk_size=120, overlap=30):

    words = text.split() # list of words

    chunks = []

    step = chunk_size - overlap

    for start in range(0, len(words), step):

        chunk = words[start:start + chunk_size] # list of words for current chunk

        if not chunk:
            break

        chunks.append(" ".join(chunk)) # Reconstructing the chunk string from list of words and append the chunk

        if start + chunk_size >= len(words):
            break

    return chunks

chunks = create_chunks(text)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Total chunks: 6

--- Chunk 0 ---
Retrieval Augmented Generation (RAG) Retrieval Augmented Generation is a technique that combines information retrieval with large language models. Instead of relying entirely on the knowledge stored in an LLM during training, RAG retrieves relevant information from an external knowledge source and provides it as context to the LLM. This helps the model answer questions using specific and potentially updated information. Document Chunking Document chunking is the process of splitting large documents into smaller pieces called chunks. Embedding models have input token limits, so processing an entire large document at once may not be practical. Smaller chunks also allow the retrieval system to identify specific portions of a document rather than retrieving the entire document. Chunk size and overlap can significantly

--- Chunk 1 ---
may not be practical. Smaller chunks also allow the retrieval system to identify specific portions of a document rather than

In [4]:
# Load embedding model
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)

model.eval()

C:\Users\SAGNICK\anaconda3\envs\mypyref\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAGNICK\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1491.85it/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.

In [6]:
!pip show huggingface-hub httpx2 brotli Brotli httpx

Name: huggingface_hub
Version: 2.4.0
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache-2.0
Location: C:\Users\SAGNICK\anaconda3\envs\mypyref\Lib\site-packages
Requires: click, filelock, fsspec, hf-xet, httpx2, packaging, pyyaml, tqdm, typing-extensions
Required-by: sentence-transformers, tokenizers, transformers
---
Name: httpx2
Version: 2.12.0
Summary: The next generation HTTP client.
Home-page: https://github.com/pydantic/httpx2
Author: 
Author-email: Tom Christie <tom@tomchristie.com>
License-Expression: BSD-3-Clause
Location: C:\Users\SAGNICK\anaconda3\envs\mypyref\Lib\site-packages
Requires: anyio, httpcore2, idna, truststore
Required-by: huggingface_hub, langsmith
---
Name: httpx
Version: 0.28.1
Summary: The next generation HTTP client.
Home-page: https://github.com/encode/httpx
Author: 
A

In [7]:
!pip uninstall -y Brotli brotlicffi

Found existing installation: brotlicffi 1.0.9.2
Uninstalling brotlicffi-1.0.9.2:
  Successfully uninstalled brotlicffi-1.0.9.2


In [5]:
import torch
import torch.nn.functional as F
import numpy as np

In [6]:
def generate_embeddings(sentences):

    # 1. Tokenization
    encoded_input = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    # 2. Transformer inference
    with torch.no_grad():
        model_output = model(**encoded_input)

    # 3. Extract contextual token embeddings
    token_embeddings = model_output.last_hidden_state

    # 4. Get attention mask
    attention_mask = encoded_input["attention_mask"]

    # 5. Expand mask to match embedding dimensions
    input_mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # 6. Mean pooling
    sum_embeddings = torch.sum(
        token_embeddings * input_mask,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask.sum(dim=1),
        min=1e-9
    )

    embeddings = sum_embeddings / sum_mask

    # 7. Normalize
    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    return embeddings.cpu().numpy()

embeddings = generate_embeddings(chunks)

print("Embedding shape:", embeddings.shape)
print("First embedding:", embeddings[0][:10])

Embedding shape: (6, 384)
First embedding: [-0.07144312  0.04029175 -0.01265628  0.04852672  0.03666174  0.01324387
 -0.01226955  0.002668    0.00162012 -0.05009174]


In [7]:
# Verify norm
norms = np.linalg.norm(embeddings, axis=1)

print(norms)

[0.99999994 1.         1.         1.         1.         1.        ]


In [8]:
# create vector db
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)
collection = client.get_or_create_collection(
    name="rag_knowledge_manual",
    metadata={"hnsw:space": "cosine"}
)

In [9]:
# generate IDs
ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
]

# generate metadata
metadatas = [
    {
        "source": "knowledge_text.txt",
        "chunk_index": i
    }
    for i in range(len(chunks))
]

#Insert everything
collection.upsert(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)

print("Stored chunks:", collection.count())

Stored chunks: 6


In [10]:
# Semantic retrieval
query = "How does semantic search find relevant information?"

query_embedding = generate_embeddings([query])

print("Query vector shape:", query_embedding.shape)

Query vector shape: (1, 384)


In [11]:
# Search for top 3 relevant chunks
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

In [12]:
for i in range(len(results["documents"][0])):

    print(f"\nResult {i+1}")

    print("Chunk:")
    print(results["documents"][0][i])

    print("\nMetadata:")
    print(results["metadatas"][0][i])

    print("\nDistance:")
    print(results["distances"][0][i])

    print("-" * 60)


Result 1
Chunk:
query is converted into an embedding, and the database finds the nearest vectors. A retrieval system often selects the top K most similar chunks. These chunks are then passed to the generation model as additional context. Metadata Filtering Metadata filtering allows a retrieval system to restrict its search to documents satisfying specific conditions. Metadata may include document names, authors, categories, dates and page numbers. For example, a system can retrieve information only from documents belonging to a particular category. Metadata filtering can be combined with vector similarity search. Limitations of RAG RAG systems can retrieve irrelevant or incomplete information. Poor chunking, weak embeddings and ambiguous queries can reduce retrieval quality. Even when relevant documents are retrieved, an LLM may misunderstand

Metadata:
{'source': 'knowledge_text.txt', 'chunk_index': 4}

Distance:
0.4337102174758911
----------------------------------------------------

In [13]:
# Prepare final answer 
retrieved_chunks = results["documents"][0]
context = "\n\n".join(retrieved_chunks)

print(context)

query is converted into an embedding, and the database finds the nearest vectors. A retrieval system often selects the top K most similar chunks. These chunks are then passed to the generation model as additional context. Metadata Filtering Metadata filtering allows a retrieval system to restrict its search to documents satisfying specific conditions. Metadata may include document names, authors, categories, dates and page numbers. For example, a system can retrieve information only from documents belonging to a particular category. Metadata filtering can be combined with vector similarity search. Limitations of RAG RAG systems can retrieve irrelevant or incomplete information. Poor chunking, weak embeddings and ambiguous queries can reduce retrieval quality. Even when relevant documents are retrieved, an LLM may misunderstand

using similarity measures such as cosine similarity. The most similar vectors are selected for retrieval. Cosine Similarity Cosine similarity measures the simil

In [14]:
# Final prompt for LLM 
prompt = f"""
You are a helpful assistant.

Answer the user's question using only
the information provided in the context.

If the answer is not available in the
context, say that you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


You are a helpful assistant.

Answer the user's question using only
the information provided in the context.

If the answer is not available in the
context, say that you don't know.

Context:
query is converted into an embedding, and the database finds the nearest vectors. A retrieval system often selects the top K most similar chunks. These chunks are then passed to the generation model as additional context. Metadata Filtering Metadata filtering allows a retrieval system to restrict its search to documents satisfying specific conditions. Metadata may include document names, authors, categories, dates and page numbers. For example, a system can retrieve information only from documents belonging to a particular category. Metadata filtering can be combined with vector similarity search. Limitations of RAG RAG systems can retrieve irrelevant or incomplete information. Poor chunking, weak embeddings and ambiguous queries can reduce retrieval quality. Even when relevant documents are retr

In [15]:
# Testing the pipeline
queries = [
    "Why do we split documents into smaller pieces?",
    "How does cosine similarity work?",
    "What information does a vector database store?",
    "What problems can occur in RAG systems?",
    "Who invented Python?"
]
for query in queries:

    query_embedding = generate_embeddings([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3,
        include=["documents", "distances"]
    )

    print("\nQUESTION:", query)

    for doc, distance in zip(
        results["documents"][0],
        results["distances"][0]
    ):
        print(f"\nDistance: {distance:.4f}")
        print(doc[:250])


QUESTION: Why do we split documents into smaller pieces?

Distance: 0.5337
Retrieval Augmented Generation (RAG) Retrieval Augmented Generation is a technique that combines information retrieval with large language models. Instead of relying entirely on the knowledge stored in an LLM during training, RAG retrieves relevant i

Distance: 0.6119
may not be practical. Smaller chunks also allow the retrieval system to identify specific portions of a document rather than retrieving the entire document. Chunk size and overlap can significantly affect retrieval quality. Text Embeddings Text embed

Distance: 0.6280
query is converted into an embedding, and the database finds the nearest vectors. A retrieval system often selects the top K most similar chunks. These chunks are then passed to the generation model as additional context. Metadata Filtering Metadata 

QUESTION: How does cosine similarity work?

Distance: 0.4785
using similarity measures such as cosine similarity. The most similar vec